In [ ]:
0# Ensure notebook runs from project root so relative paths like `config/config.yaml` resolve
import os
from pathlib import Path
p = Path().resolve()
project_root = None
for parent in [p] + list(p.parents):
    if (parent / "config" / "config.yaml").exists():
        project_root = parent
        break
if project_root:
    os.chdir(str(project_root))
    print("Set notebook cwd to project root:", project_root)
else:
    print("Project root with config/config.yaml not found; cwd unchanged:", p)


In [34]:
import os


In [35]:
%pwd


'C:\\Users\\banot\\text-summarizer-'

In [36]:
os.chdir("../")

In [37]:
%pwd

'C:\\Users\\banot'

In [38]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [39]:

from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [40]:
        
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [41]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import evaluate
import torch
import pandas as pd
from tqdm import tqdm

In [42]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    
    def generate_batch_sized_chunks(self,list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    
    def calculate_metric_on_test_ds(self,dataset, metric, model, tokenizer, 
                               batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu", 
                               column_text="article", 
                               column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):
            
            inputs = tokenizer(article_batch, max_length=1024,  truncation=True, 
                            padding="max_length", return_tensors="pt")
            
            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                            attention_mask=inputs["attention_mask"].to(device), 
                            length_penalty=0.8, num_beams=8, max_length=128)
            ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''
            
            # Finally, we decode the generated texts, 
            # replace the  token, and add the decoded texts with the references to the metric.
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, 
                                    clean_up_tokenization_spaces=True) 
                for s in summaries]      
            
            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]
            
            
            metric.add_batch(predictions=decoded_summaries, references=target_batch)
            
        #  Finally compute and return the ROUGE scores.
        score = metric.compute()
        return score


    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)


        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
  
        rouge_metric = evaluate.load('rouge')

        score = self.calculate_metric_on_test_ds(
        dataset_samsum_pt['test'][0:10], rouge_metric, model_pegasus, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
            )

        rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )

        df = pd.DataFrame(rouge_dict, index = ['pegasus'] )
        df.to_csv(self.config.metric_file_name, index=False)

        

In [47]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-07-30 10:59:56,095: INFO: common: YAML file loaded successfully: config\config.yaml]
[2026-07-30 10:59:56,100: INFO: common: YAML file loaded successfully: params.yaml]
[2026-07-30 10:59:56,104: INFO: common: Created directory at: artifacts]
[2026-07-30 10:59:56,105: INFO: common: Created directory at: artifacts/model_evaluation]


c:\Users\banot\anaconda3\envs\textS\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\banot\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google

ImportError: To be able to use evaluate-metric/rouge, you need to install the following dependencies['absl', 'rouge_score', 'nltk'] using 'pip install absl rouge_score nltk' for instance'

In [44]:
# Ensure `evaluate` is installed, fix cwd, and try loading ConfigurationManager for diagnostics
import sys, subprocess, os
try:
    import evaluate
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "evaluate"])
    import evaluate
print("evaluate version:", evaluate.__version__)

print("cwd before:", os.getcwd())
# set to project root
os.chdir(r"C:\Users\banot\text-summarizer-")
print("cwd after:", os.getcwd())

from textsummarizer.constants import CONFIG_FILE_PATH
import os as _os
print("CONFIG_FILE_PATH resolved:", CONFIG_FILE_PATH.resolve())
print("exists:", _os.path.exists(CONFIG_FILE_PATH))

try:
    config = ConfigurationManager()
    print("Configuration loaded. artifacts_root:", config.config.artifacts_root)
except Exception as e:
    import traceback
    traceback.print_exc()


evaluate version: 0.4.6
cwd before: C:\Users\banot
cwd after: C:\Users\banot\text-summarizer-
CONFIG_FILE_PATH resolved: C:\Users\banot\text-summarizer-\config\config.yaml
exists: True
[2026-07-30 10:58:27,069: INFO: common: YAML file loaded successfully: config\config.yaml]
[2026-07-30 10:58:27,072: INFO: common: YAML file loaded successfully: params.yaml]
[2026-07-30 10:58:27,073: INFO: common: Created directory at: artifacts]
Configuration loaded. artifacts_root: artifacts
